# Black–Scholes Simulation + Neural-Network Trading Agent

**A self-contained notebook for Google Colab.**

This notebook does four things:

1. **Simulates** the price dynamics that underpin the **Black–Scholes (BS)** model
   (Geometric Brownian Motion) and implements the closed-form BS option pricer and
   its *Greeks*.
2. **Builds a neural network** that learns a **trading decision policy**
   (go long / stay flat / go short) from features engineered around the BS /
   volatility framework.
3. **Backtests** the resulting algorithm on **real historical market data**.
4. **Reports the results as charts** (equity curve, drawdown, Sharpe, signal
   distribution, etc.).

Everything is written in English and the relevant **theory is included inline**.

---

### Table of contents
1. [Setup & imports](#setup)
2. [Part I — Black–Scholes theory](#bs-theory)
3. [Part I — Geometric Brownian Motion simulation](#gbm)
4. [Part I — Black–Scholes pricing & Greeks](#bs-pricing)
5. [Part II — Neural-network trading: theory](#nn-theory)
6. [Part II — Real historical data](#data)
7. [Part II — Feature engineering (BS / volatility based)](#features)
7b. [Part II — Strategy configuration (the levers)](#config)
7c. [Part II — Classic trading strategies as signals](#strategies)
7d. [Part II — Chart & candlestick pattern recognition](#patterns)
7e. [Part II — Analogue (k-NN) memory of the past](#analog)
7f. [Part II — Events, news & geopolitics awareness](#events)
8. [Part II — Building the datasets: tabular & sequences](#datasets)
9. [Part II — A zoo of prediction models](#zoo)
10. [Part III — Backtesting all models](#backtest)
11. [Part III — Results & charts](#results)
11b. [Part III — Feature importance (which signals matter)](#importance)
11c. [Part III — Practical report: a €200 budget](#budget)
12. [Part IV — A portfolio of several derivatives](#portfolio)
13. [Conclusions & caveats](#conclusions)

> ⚠️ **Disclaimer.** This notebook is for **education and research** only. It is
> *not* financial advice. Backtested performance does not guarantee future
> results, and the model deliberately keeps things simple for clarity.


<a id="setup"></a>
## 1. Setup & imports

Run this cell first. In Colab, `numpy`, `pandas`, `matplotlib`, `scipy`,
`scikit-learn` and `tensorflow` are already installed; we only need to add
`yfinance` for downloading real market data.


In [ ]:
# Install the dependencies Colab may be missing.
# (Safe to re-run; each is a no-op if already installed.)
#   yfinance -> real market data (OHLCV + the VIX event/geopolitics proxy)
#   arch     -> GARCH(1,1) volatility model used by one of the trading signals
import sys, subprocess
for pkg, mod in [("yfinance", "yfinance"), ("arch", "arch")]:
    try:
        __import__(mod)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
np.random.seed(SEED)

# TensorFlow / Keras
import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow import keras
from tensorflow.keras import layers

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("NumPy   :", np.__version__)
print("Pandas  :", pd.__version__)
print("TF/Keras:", tf.__version__)


<a id="bs-theory"></a>
## 2. Part I — Black–Scholes theory

### 2.1 The model of the underlying

The Black–Scholes framework assumes the price of the underlying asset
$S_t$ follows a **Geometric Brownian Motion (GBM)**:

$$
dS_t = \mu\, S_t\, dt + \sigma\, S_t\, dW_t,
$$

where

- $\mu$ is the (real-world) **drift** / expected return,
- $\sigma$ is the **volatility**,
- $W_t$ is a standard **Brownian motion** (Wiener process).

Applying Itô's lemma to $\ln S_t$ gives the closed-form solution

$$
S_t = S_0 \, \exp\!\Big[\big(\mu - \tfrac12\sigma^2\big)t + \sigma W_t\Big],
$$

so **log-returns are normally distributed** and prices are **log-normal**.

### 2.2 The Black–Scholes PDE

Under a no-arbitrage argument with continuous **delta-hedging**, the value
$V(S,t)$ of any European derivative satisfies the **Black–Scholes PDE**:

$$
\frac{\partial V}{\partial t}
+ \tfrac12\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}
+ r S \frac{\partial V}{\partial S}
- r V = 0,
$$

where $r$ is the **risk-free rate**. Note the drift $\mu$ disappears: pricing is
done under the **risk-neutral measure** where the asset drifts at $r$.

### 2.3 Closed-form European option prices

For a European **call** ($C$) and **put** ($P$) with strike $K$ and time to
maturity $T$:

$$
d_1 = \frac{\ln(S/K) + (r + \tfrac12\sigma^2)T}{\sigma\sqrt{T}},
\qquad
d_2 = d_1 - \sigma\sqrt{T},
$$

$$
C = S\,\Phi(d_1) - K e^{-rT}\,\Phi(d_2),
\qquad
P = K e^{-rT}\,\Phi(-d_2) - S\,\Phi(-d_1),
$$

where $\Phi$ is the standard normal CDF. Put–call parity holds:
$C - P = S - K e^{-rT}$.

### 2.4 The Greeks

The **Greeks** are sensitivities of the option value used for hedging and risk.
For a call:

| Greek | Meaning | Formula |
|-------|---------|---------|
| $\Delta$ | $\partial V/\partial S$ | $\Phi(d_1)$ |
| $\Gamma$ | $\partial^2 V/\partial S^2$ | $\dfrac{\phi(d_1)}{S\sigma\sqrt{T}}$ |
| $\mathcal{V}$ (Vega) | $\partial V/\partial \sigma$ | $S\,\phi(d_1)\sqrt{T}$ |
| $\Theta$ | $\partial V/\partial t$ | $-\dfrac{S\phi(d_1)\sigma}{2\sqrt{T}} - rKe^{-rT}\Phi(d_2)$ |
| $\rho$ | $\partial V/\partial r$ | $KTe^{-rT}\Phi(d_2)$ |

$\phi$ is the standard normal PDF. **Delta** is the key link to trading: it is the
number of units of the underlying needed to hedge the option, and — as we'll use
later — a natural, bounded way to translate a *directional forecast* into a
*position size*.


<a id="gbm"></a>
## 3. Part I — Simulating Geometric Brownian Motion

We now simulate GBM paths (the "world" the BS model assumes) and check that the
empirical distribution of log-returns matches the theory.


In [ ]:
def simulate_gbm(S0, mu, sigma, T, n_steps, n_paths, seed=None):
    '''Simulate Geometric Brownian Motion paths.

    dS = mu*S*dt + sigma*S*dW  ->  exact log-Euler scheme.

    Returns
    -------
    t     : (n_steps+1,) time grid
    paths : (n_paths, n_steps+1) simulated price paths
    '''
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    # Brownian increments
    dW = rng.normal(0.0, np.sqrt(dt), size=(n_paths, n_steps))
    # Log-return increments (exact solution of GBM)
    incr = (mu - 0.5 * sigma**2) * dt + sigma * dW
    log_paths = np.concatenate(
        [np.zeros((n_paths, 1)), np.cumsum(incr, axis=1)], axis=1
    )
    paths = S0 * np.exp(log_paths)
    t = np.linspace(0.0, T, n_steps + 1)
    return t, paths


# Parameters
S0, mu, sigma, T = 100.0, 0.08, 0.20, 1.0
n_steps, n_paths = 252, 200

t, paths = simulate_gbm(S0, mu, sigma, T, n_steps, n_paths, seed=SEED)
print("Simulated", paths.shape[0], "paths over", paths.shape[1], "time points.")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# (a) A sample of simulated paths
for i in range(40):
    ax[0].plot(t, paths[i], lw=0.8, alpha=0.6)
ax[0].plot(t, S0 * np.exp(mu * t), color="black", lw=2.5, label=r"$E[S_t]=S_0e^{\mu t}$")
ax[0].set_title("Simulated GBM price paths")
ax[0].set_xlabel("Time (years)"); ax[0].set_ylabel("Price"); ax[0].legend()

# (b) Terminal log-returns vs the theoretical normal density
terminal_log_ret = np.log(paths[:, -1] / S0)
ax[1].hist(terminal_log_ret, bins=30, density=True, alpha=0.6, label="Simulated")
xs = np.linspace(terminal_log_ret.min(), terminal_log_ret.max(), 200)
theo = norm.pdf(xs, (mu - 0.5 * sigma**2) * T, sigma * np.sqrt(T))
ax[1].plot(xs, theo, "r-", lw=2, label="Theoretical N")
ax[1].set_title("Terminal log-returns vs Black–Scholes theory")
ax[1].set_xlabel(r"$\ln(S_T/S_0)$"); ax[1].set_ylabel("Density"); ax[1].legend()

plt.tight_layout(); plt.show()


<a id="bs-pricing"></a>
## 4. Part I — Black–Scholes pricing & Greeks

We implement the closed-form pricer and the Greeks, then visualise how the call
price and its Delta behave across the moneyness spectrum.


In [ ]:
def bs_price(S, K, T, r, sigma, option="call"):
    '''Black–Scholes price of a European option (vectorised).'''
    S, K, T, sigma = map(np.asarray, (S, K, T, sigma))
    T = np.maximum(T, 1e-12); sigma = np.maximum(sigma, 1e-12)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


def bs_greeks(S, K, T, r, sigma, option="call"):
    '''Return a dict with Delta, Gamma, Vega, Theta, Rho.'''
    S, K, T, sigma = map(np.asarray, (S, K, T, sigma))
    T = np.maximum(T, 1e-12); sigma = np.maximum(sigma, 1e-12)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    pdf = norm.pdf(d1)
    if option == "call":
        delta = norm.cdf(d1)
        theta = (-S * pdf * sigma / (2 * np.sqrt(T))
                 - r * K * np.exp(-r * T) * norm.cdf(d2))
        rho = K * T * np.exp(-r * T) * norm.cdf(d2)
    else:
        delta = norm.cdf(d1) - 1.0
        theta = (-S * pdf * sigma / (2 * np.sqrt(T))
                 + r * K * np.exp(-r * T) * norm.cdf(-d2))
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)
    gamma = pdf / (S * sigma * np.sqrt(T))
    vega = S * pdf * np.sqrt(T)
    return {"delta": delta, "gamma": gamma, "vega": vega, "theta": theta, "rho": rho}


# Sanity check: put-call parity  C - P = S - K e^{-rT}
K, r = 100.0, 0.03
C = bs_price(100, K, 1.0, r, 0.2, "call")
P = bs_price(100, K, 1.0, r, 0.2, "put")
print(f"Call = {C:.4f}, Put = {P:.4f}")
print(f"C - P = {C - P:.4f}  vs  S - K e^-rT = {100 - K*np.exp(-r*1.0):.4f}")


In [ ]:
S_grid = np.linspace(60, 140, 200)
call_prices = bs_price(S_grid, K, T=0.5, r=r, sigma=0.2, option="call")
greeks = bs_greeks(S_grid, K, T=0.5, r=r, sigma=0.2, option="call")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(S_grid, call_prices, lw=2)
ax[0].axvline(K, color="gray", ls="--", label="Strike K")
ax[0].plot(S_grid, np.maximum(S_grid - K, 0), "k:", label="Payoff at maturity")
ax[0].set_title("Black–Scholes call price"); ax[0].set_xlabel("Spot S")
ax[0].set_ylabel("Option value"); ax[0].legend()

ax[1].plot(S_grid, greeks["delta"], lw=2, label=r"$\Delta$ (call)")
ax[1].plot(S_grid, greeks["gamma"] * 10, lw=2, label=r"$\Gamma \times 10$")
ax[1].axvline(K, color="gray", ls="--")
ax[1].set_title("Delta & Gamma vs spot"); ax[1].set_xlabel("Spot S")
ax[1].set_ylabel("Greek value"); ax[1].legend()
plt.tight_layout(); plt.show()


<a id="nn-theory"></a>
## 5. Part II — Neural-network trading: theory

### 5.1 Idea

Classical Black–Scholes assumes **constant, known volatility** and prices
derivatives; it does **not** tell you *which direction* the market will move.
Here we take the complementary view: we keep the **volatility/return machinery of
the BS world** and let a **neural network learn a directional decision** from data.

The pipeline is:

$$
\underbrace{\text{market data}}_{\text{prices}}
\;\rightarrow\;
\underbrace{\text{BS / volatility features}}_{\text{returns, }\sigma,\text{ z-scores, Greeks}}
\;\rightarrow\;
\underbrace{\text{neural network}}_{\text{classifier}}
\;\rightarrow\;
\underbrace{\text{position}}_{\text{long / flat / short}}
\;\rightarrow\;
\underbrace{\text{backtest}}_{\text{P\&L, Sharpe}}
$$

### 5.2 What the network predicts

We frame trading as a **3-class classification** of the **next day's return**:

- class **+1 (Long)**  if next-day return $> +\tau$,
- class **0  (Flat)**  if $|$next-day return$| \le \tau$,
- class **−1 (Short)** if next-day return $< -\tau$,

with a small **dead-band** $\tau$ (a fraction of daily volatility) so the model
is not forced to bet on noise. The network outputs class probabilities via a
**softmax**; the trading position is a **volatility-scaled, Delta-like mapping**
of those probabilities into $[-1, +1]$.

### 5.3 Why Black–Scholes features?

- **Realized volatility** $\sigma$ is *the* BS parameter and strongly drives
  risk-adjusted returns; we feed several horizons of it.
- **Standardized moves** $z = r_t / \sigma_t$ are exactly the argument of the
  normal distribution in the BS formula — natural, scale-free features.
- A synthetic **BS Delta** built from a rolling z-score gives a smooth, bounded
  "how far in/out of the money is momentum" signal.

### 5.4 Avoiding look-ahead bias

Financial ML is easy to get wrong. We are careful to:

- build every feature from **past** data only (rolling windows, then `shift`),
- split **chronologically** (train → validation → test, never shuffled),
- fit the scaler on the **training set only**,
- apply realistic **transaction costs** in the backtest.


<a id="data"></a>
## 6. Part II — Real historical data

We download real daily prices with `yfinance`. If the Colab runtime has no
internet access (or the download fails), we **fall back to a GBM-simulated
series** so the whole notebook still runs end-to-end.


In [ ]:
TICKER   = "SPY"          # try e.g. "AAPL", "MSFT", "^GSPC", "BTC-USD"
START    = "2010-01-01"
END      = "2024-12-31"

OHLCV = ["Open", "High", "Low", "Close", "Volume"]

def load_prices(ticker, start, end):
    '''Return an OHLCV DataFrame (Open/High/Low/Close/Volume), real or simulated.

    OHLCV is needed by the candlestick patterns, the Stochastic / Keltner
    signals and the volume-based event proxies added later.
    '''
    try:
        import yfinance as yf
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if df is not None and len(df) > 250:
            out = df[OHLCV].dropna().copy()
            out.attrs["source"] = f"real data ({ticker})"
            return out
        raise ValueError("empty download")
    except Exception as e:
        print(f"[warn] download failed ({e}); using GBM-simulated OHLCV fallback.")
        n = 252 * 12
        _, p = simulate_gbm(100.0, 0.07, 0.18, n / 252, n, 1, seed=SEED)
        close = p[0]
        rng = np.random.default_rng(SEED)
        # Synthesize a plausible intraday range and volume around the close
        rel = np.abs(rng.normal(0, 0.008, size=close.shape))
        high = close * (1 + rel)
        low = close * (1 - rel)
        openp = np.concatenate([[close[0]], close[:-1]]) * (1 + rng.normal(0, 0.004, close.shape))
        vol = rng.lognormal(15, 0.4, size=close.shape)
        idx = pd.bdate_range(start=start, periods=n + 1)
        out = pd.DataFrame({"Open": openp, "High": np.maximum(high, np.maximum(openp, close)),
                            "Low": np.minimum(low, np.minimum(openp, close)),
                            "Close": close, "Volume": vol}, index=idx)
        out.attrs["source"] = "SIMULATED (offline fallback)"
        return out

prices = load_prices(TICKER, START, END)
SOURCE = prices.attrs.get("source", "unknown")
print("Data source:", SOURCE)
print("Rows:", len(prices), "| from", prices.index[0].date(), "to", prices.index[-1].date())
prices.tail()


In [ ]:
plt.figure()
plt.plot(prices.index, prices["Close"], lw=1.2)
plt.title(f"{TICKER} closing price  [{SOURCE}]")
plt.xlabel("Date"); plt.ylabel("Price"); plt.tight_layout(); plt.show()


<a id="features"></a>
## 7. Part II — Feature engineering (BS / volatility based)

Every feature below is computed from **past** information only. The realized
volatility is the annualized standard deviation of log-returns — the empirical
counterpart of the BS $\sigma$.


In [ ]:
def build_features(prices, vol_windows=(5, 10, 21, 63), mom_windows=(5, 10, 21, 63)):
    df = pd.DataFrame(index=prices.index)
    df["close"] = prices["Close"]
    df["log_ret"] = np.log(df["close"]).diff()

    # --- Realized (annualized) volatility over several horizons: the BS sigma ---
    for w in vol_windows:
        df[f"vol_{w}"] = df["log_ret"].rolling(w).std() * np.sqrt(252)

    # --- Momentum / trend features ---
    for w in mom_windows:
        df[f"mom_{w}"] = df["close"].pct_change(w)

    # --- Standardized daily move  z = r_t / sigma_t  (BS-normal argument) ---
    df["z_score"] = df["log_ret"] / (df["log_ret"].rolling(21).std() + 1e-9)

    # --- Distance from moving averages (moneyness-like) ---
    for w in (21, 63):
        ma = df["close"].rolling(w).mean()
        df[f"dist_ma_{w}"] = (df["close"] - ma) / ma

    # --- RSI(14): a bounded momentum oscillator ---
    delta = df["close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-9)
    df["rsi_14"] = 100 - 100 / (1 + rs)

    # --- Synthetic Black–Scholes Delta of an ATM call whose "moneyness" is
    #     driven by the trailing z-score momentum. Smooth, bounded directional
    #     signal in (0, 1); 0.5 = neutral. ---
    roll_z = df["z_score"].rolling(10).mean().fillna(0.0)
    sig = df["vol_21"].fillna(df["vol_21"].median()).clip(0.05, 1.0)
    S_syn = 100.0 * np.exp(0.02 * roll_z)          # momentum tilts the spot
    df["bs_delta"] = bs_greeks(S_syn.values, 100.0, 0.25, 0.02, sig.values, "call")["delta"]
    df["bs_gamma"] = bs_greeks(S_syn.values, 100.0, 0.25, 0.02, sig.values, "call")["gamma"]

    return df

feat = build_features(prices)
FEATURE_COLS = [c for c in feat.columns if c not in ("close", "log_ret")]
print("Features:", FEATURE_COLS)
feat[FEATURE_COLS].tail()


<a id="config"></a>
## 7b. Strategy configuration — the levers

Before labelling and training we expose the **decision levers** in one config
dictionary, so the whole strategy can be re-tuned from a single place. These are
exactly the improvements motivated by the earlier flat-equity result:

| Lever | Meaning | Why it matters |
|-------|---------|----------------|
| `horizon` | forecast the **H-day-ahead** direction (not just tomorrow) | longer horizons have a *higher signal-to-noise ratio* than 1-day noise |
| `lookback` | sequence length fed to LSTM / GRU / CNN / Transformer | how much history the sequence models see |
| `gain` | amplifies conviction → **larger positions** | fixes the "barely invested" problem (positions were ~0.1) |
| `long_bias` | structural tilt toward being invested | equities **drift up**; a long tilt usually beats symmetric long/short on indices |
| `allow_short` | if `False`, the strategy is **long-or-flat** | avoids fighting the market's upward drift |
| `vol_target` | annualized volatility target for sizing | the BS $\sigma$ entering position sizing (risk control) |
| `max_leverage` | cap on absolute exposure | how aggressive we allow the book to get |
| `cost_bps` | transaction cost per unit of turnover | keeps the backtest honest |


In [ ]:
# --- The strategy levers, all in one place ---------------------------------
CONFIG = {
    "horizon":      5,      # predict the sign of the 5-day-ahead return (multi-day)
    "lookback":     20,     # sequence length for LSTM/GRU/CNN/Transformer
    "deadband":     0.5,    # dead-band as a fraction of H-day volatility
    "gain":         4.0,    # amplify conviction  ->  larger positions
    "long_bias":    0.15,   # structural tilt toward being invested (drift up)
    "allow_short":  False,  # LONG-BIAS regime: long-or-flat, no shorting
    "vol_target":   0.15,   # annualized vol target for position sizing
    "max_leverage": 1.5,    # cap on |exposure|
    "cost_bps":     1.0,    # per-trade cost in basis points of turnover
}
CONFIG


<a id="strategies"></a>
## 7c. Part II — Classic trading strategies as signals

Instead of only hand-made statistical features, we now compute the **signals of
well-known rule-based trading strategies** and feed them to the network as extra
inputs. The neural network then acts as a **meta-model** that learns how much to
trust each strategy in each market state (a form of *stacking*).

Every signal is in $\{-1, 0, +1\}$ (short / flat / long) or a bounded continuous
value, computed **causally** (only past data), so it is safe to use as a feature.

| Family | Strategy | Rule (long condition) |
|--------|----------|-----------------------|
| Trend | **SMA/EMA crossover** | fast MA (20) above slow MA (50) |
| Trend | **MACD** | MACD line above its signal line |
| Trend | **Donchian breakout** | new 20-day high (short on new low) |
| Trend | **Time-series momentum** | positive trailing 63-day return |
| Mean-rev | **Bollinger Bands** | price below lower band (short above upper) |
| Mean-rev | **RSI(14)** | oversold RSI < 30 (short if > 70) |
| Mean-rev | **Z-score reversion** | fade the standardized deviation from the mean |
| Mean-rev | **Stochastic** | %D below 20 (short above 80) |
| Vol / BS | **Volatility regime** | realized $\sigma$ below its median (risk-on) |
| Vol / BS | **BS-Delta momentum** | synthetic Black–Scholes Delta above 0.5 |
| Vol / BS | **ATR / Keltner breakout** | close above the upper Keltner band |
| Vol / BS | **GARCH(1,1) regime** | model-forecast volatility below its median |


In [ ]:
# --- Rule-based trading-strategy signals (all causal) ----------------------
def sig_sma_cross(close, fast=20, slow=50):
    return np.sign(close.rolling(fast).mean() - close.rolling(slow).mean()).fillna(0.0)

def sig_macd(close, f=12, s=26, sig=9):
    macd = close.ewm(span=f, adjust=False).mean() - close.ewm(span=s, adjust=False).mean()
    signal = macd.ewm(span=sig, adjust=False).mean()
    return np.sign(macd - signal).fillna(0.0)

def sig_donchian(close, n=20):
    hi, lo = close.rolling(n).max(), close.rolling(n).min()
    s = pd.Series(np.nan, index=close.index)
    s[close >= hi] = 1.0; s[close <= lo] = -1.0
    return s.ffill().fillna(0.0)

def sig_tsmom(close, n=63):
    return np.sign(close.pct_change(n)).fillna(0.0)

def sig_bollinger(close, n=20, k=2.0):
    ma, sd = close.rolling(n).mean(), close.rolling(n).std()
    z = (close - ma) / (sd + 1e-9)
    s = pd.Series(0.0, index=close.index)
    s[z < -k] = 1.0; s[z > k] = -1.0
    return s

def sig_rsi(close, n=14, lo=30, hi=70):
    d = close.diff()
    g = d.clip(lower=0).rolling(n).mean(); l = (-d.clip(upper=0)).rolling(n).mean()
    rsi = 100 - 100 / (1 + g / (l + 1e-9))
    s = pd.Series(0.0, index=close.index)
    s[rsi < lo] = 1.0; s[rsi > hi] = -1.0
    return s

def sig_zscore(close, n=21):
    ma, sd = close.rolling(n).mean(), close.rolling(n).std()
    z = (close - ma) / (sd + 1e-9)
    return np.clip(-z, -1.0, 1.0).fillna(0.0)          # fade deviations (mean reversion)

def sig_stochastic(high, low, close, n=14, d=3, lo=20, hi=80):
    ll, hh = low.rolling(n).min(), high.rolling(n).max()
    k = 100 * (close - ll) / ((hh - ll) + 1e-9)
    dline = k.rolling(d).mean()
    s = pd.Series(0.0, index=close.index)
    s[dline < lo] = 1.0; s[dline > hi] = -1.0
    return s

def sig_volregime(close, n=21):
    rv = np.log(close).diff().rolling(n).std()
    med = rv.rolling(252, min_periods=60).median()
    s = pd.Series(0.0, index=close.index)
    s[rv < med] = 1.0; s[rv > med] = -1.0
    return s.fillna(0.0)

def sig_bsdelta_mom(bs_delta):
    return np.sign(bs_delta - 0.5).fillna(0.0)

def sig_atr_keltner(high, low, close, n=20, mult=1.5):
    prev = close.shift()
    tr = pd.concat([(high - low), (high - prev).abs(), (low - prev).abs()], axis=1).max(axis=1)
    atr = tr.rolling(n).mean(); ma = close.rolling(n).mean()
    s = pd.Series(np.nan, index=close.index)
    s[close > ma + mult * atr] = 1.0; s[close < ma - mult * atr] = -1.0
    return s.ffill().fillna(0.0)

def sig_garch(close):
    try:
        from arch import arch_model
        r = (np.log(close).diff().dropna()) * 100.0
        res = arch_model(r, mean="Constant", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
        cv = res.conditional_volatility.reindex(close.index)
        med = cv.rolling(252, min_periods=60).median()
        s = pd.Series(0.0, index=close.index)
        s[cv < med] = 1.0; s[cv > med] = -1.0
        return s.fillna(0.0)
    except Exception as e:
        print(f"[warn] GARCH signal unavailable ({e}); using neutral 0.")
        return pd.Series(0.0, index=close.index)

o, h, l, c = prices["Open"], prices["High"], prices["Low"], prices["Close"]
STRAT = pd.DataFrame({
    "strat_sma":      sig_sma_cross(c),
    "strat_macd":     sig_macd(c),
    "strat_donchian": sig_donchian(c),
    "strat_tsmom":    sig_tsmom(c),
    "strat_boll":     sig_bollinger(c),
    "strat_rsi":      sig_rsi(c),
    "strat_zscore":   sig_zscore(c),
    "strat_stoch":    sig_stochastic(h, l, c),
    "strat_volreg":   sig_volregime(c),
    "strat_bsdelta":  sig_bsdelta_mom(feat["bs_delta"]),
    "strat_keltner":  sig_atr_keltner(h, l, c),
    "strat_garch":    sig_garch(c),
})
STRAT_COLS = list(STRAT.columns)
for col in STRAT_COLS:
    feat[col] = STRAT[col].reindex(feat.index)
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS + STRAT_COLS))
print(f"Added {len(STRAT_COLS)} strategy signals. Total features: {len(FEATURE_COLS)}")
STRAT.tail()


<a id="patterns"></a>
## 7d. Part II — Chart & candlestick pattern recognition

Technical traders react to **price patterns**. We detect a handful of classic
**candlestick / chart patterns** directly from the OHLC bars and expose them as
features, so the network can learn whether a "hammer" or a "bullish engulfing"
actually carries predictive information for this asset.

- **Doji** — tiny body: indecision.
- **Hammer** — long lower shadow: potential bullish reversal.
- **Bullish / Bearish engulfing** — today's body engulfs yesterday's: reversal.
- **Gap up / down** — open jumps away from the previous close (reaction to news).
- **Inside bar** — today's range inside yesterday's: compression before a move.
- **Trend structure** — net count of higher-highs / higher-lows (up-trend health).


In [ ]:
# --- Candlestick / chart pattern features (causal, from OHLC) ---------------
def build_patterns(prices):
    o, h, l, c = prices["Open"], prices["High"], prices["Low"], prices["Close"]
    body = c - o
    rng = (h - l) + 1e-9
    up_shadow = h - np.maximum(o, c)
    dn_shadow = np.minimum(o, c) - l
    prev_c, prev_o = c.shift(), o.shift()

    pat = pd.DataFrame(index=prices.index)
    pat["pat_doji"]   = (body.abs() <= 0.1 * rng).astype(float)
    pat["pat_hammer"] = ((dn_shadow >= 2 * body.abs()) & (up_shadow <= 0.3 * rng)).astype(float)
    pat["pat_engulf"] = (
        ((c > o) & (prev_c < prev_o) & (c >= prev_o) & (o <= prev_c)).astype(float)   # bullish +1
        - ((c < o) & (prev_c > prev_o) & (o >= prev_c) & (c <= prev_o)).astype(float) # bearish -1
    )
    gap = o / prev_c - 1.0
    pat["pat_gap"]    = (np.sign(gap) * (gap.abs() > 0.005)).fillna(0.0)
    pat["pat_inside"] = ((h < h.shift()) & (l > l.shift())).astype(float)
    hh = (h > h.shift()).astype(float); hl = (l > l.shift()).astype(float)
    pat["pat_trend"]  = (hh.rolling(5).mean() + hl.rolling(5).mean() - 1.0).fillna(0.0)
    return pat.fillna(0.0)

PATTERNS = build_patterns(prices)
PATTERN_COLS = list(PATTERNS.columns)
for col in PATTERN_COLS:
    feat[col] = PATTERNS[col].reindex(feat.index)
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS + PATTERN_COLS))
print(f"Added {len(PATTERN_COLS)} pattern features. Total features: {len(FEATURE_COLS)}")
print("Pattern hit-rate (share of days flagged):")
print((PATTERNS.abs() > 0).mean().round(3).to_string())


<a id="analog"></a>
## 7e. Part II — Learning from the past: an analogue (k-NN) memory

"Has the market looked like *this* before, and what happened next?" We give the
model an explicit **memory of history**: for each day we find the **k most similar
past days** (nearest neighbours in feature space) and summarise what happened
**after** them. This is *analogue forecasting* / case-based reasoning.

Two safeguards keep it honest:

- **Causality** — a query day only searches days that are *strictly in its past*.
- **Purging** — neighbours must be old enough that their `horizon`-day outcome was
  already known before the query day (no leakage of the future).

The memory produces two features: the neighbours' **average forward return** and
their **win rate** (share of positive outcomes).


In [ ]:
# --- Analogue k-NN memory: causal, purged --------------------------------
from sklearn.neighbors import NearestNeighbors

def analogue_memory(feat, state_cols, horizon, k=50, step=21, min_hist=252):
    '''For each day, average forward outcome of its k nearest PAST neighbours.'''
    idx = feat.index
    # Causal z-scoring of the state (expanding stats -> no look-ahead)
    S = feat[state_cols].astype(float)
    mu = S.expanding(min_periods=30).mean()
    sd = S.expanding(min_periods=30).std() + 1e-9
    Z = ((S - mu) / sd).fillna(0.0).values
    fwd = np.log(feat["close"].shift(-horizon) / feat["close"]).values  # outcome per row

    n = len(idx)
    a_ret = np.full(n, np.nan)
    a_win = np.full(n, np.nan)
    for b in range(min_hist, n, step):
        lib_end = b - horizon                      # purge: outcomes known before b
        if lib_end < 30:
            continue
        mask = np.isfinite(fwd[:lib_end]) & np.isfinite(Z[:lib_end]).all(axis=1)
        if mask.sum() < k + 5:
            continue
        lib_Z = Z[:lib_end][mask]; lib_y = fwd[:lib_end][mask]
        nn = NearestNeighbors(n_neighbors=k).fit(lib_Z)
        q = slice(b, min(b + step, n))
        _, ind = nn.kneighbors(Z[q])
        a_ret[q] = lib_y[ind].mean(axis=1)
        a_win[q] = (lib_y[ind] > 0).mean(axis=1)
    out = pd.DataFrame({"mem_analog_ret": a_ret, "mem_analog_win": a_win}, index=idx)
    return out

STATE_COLS = [c for c in ["vol_21", "mom_21", "mom_5", "z_score", "rsi_14", "dist_ma_21"]
              if c in feat.columns]
MEMORY = analogue_memory(feat, STATE_COLS, horizon=CONFIG["horizon"], k=50)
MEM_COLS = list(MEMORY.columns)
for col in MEM_COLS:
    feat[col] = MEMORY[col]
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS + MEM_COLS))
print(f"Analogue memory built from {len(STATE_COLS)} state features: {STATE_COLS}")
print(f"Added {len(MEM_COLS)} memory features. Total features: {len(FEATURE_COLS)}")
feat[MEM_COLS].dropna().tail()


<a id="events"></a>
## 7f. Part II — Events, news & geopolitics awareness

Markets move on **events**: geopolitical shocks, macro releases and
**stock-specific news** (earnings, dividends, surprises). We add three honest,
**backtestable** proxies for this — real, historical, and leak-free:

1. **Market-fear proxy (geopolitics/macro):** the **VIX** index. Spikes in the
   VIX are the market's own measure of stress from geopolitical / macro events.
2. **Curated event overlay:** a small, **editable table of major geopolitical /
   macro events** (crashes, wars, rate shocks, elections). Each becomes a
   **decaying shock** feature — strong right after the event, fading over weeks.
3. **Stock-specific event proxies:** **abnormal volume**, **overnight gaps**, and
   **earnings / dividend proximity** — the fingerprints that *news left in the
   price and volume tape*, available for every historical day.

> **On live news sentiment.** True real-time headline sentiment needs an external
> news API (with a key), which a public backtest cannot replay historically
> without look-ahead. We therefore expose a **`news_sentiment_hook`** you can plug
> your own feed into; by default it is neutral. The proxies above already capture
> *the market's reaction* to news, which is what a price model can actually use.


In [ ]:
# --- (1) VIX: market-fear / geopolitics proxy ------------------------------
def load_vix(start, end, index_like):
    try:
        import yfinance as yf
        v = yf.download("^VIX", start=start, end=end, progress=False, auto_adjust=True)
        if isinstance(v.columns, pd.MultiIndex):
            v.columns = v.columns.get_level_values(0)
        vix = v["Close"].reindex(index_like).ffill()
        if vix.notna().mean() > 0.5:
            return vix, "real ^VIX"
        raise ValueError("empty VIX")
    except Exception as e:
        print(f"[warn] VIX download failed ({e}); using realized-vol proxy.")
        rv = np.log(prices["Close"]).diff().rolling(21).std() * np.sqrt(252) * 100
        return rv.reindex(index_like).ffill(), "realized-vol proxy"

vix, vix_src = load_vix(START, END, feat.index)
print("VIX source:", vix_src)

# --- (2) Curated geopolitical / macro event overlay (EDIT ME) --------------
# (date, label, severity 1-3).  Add your own rows; only dates within the sample
# are used.  Turned into a decaying 'shock' that fades with a ~15-day half-life.
GEO_EVENTS = [
    ("2010-05-06", "Flash Crash", 2),
    ("2011-08-05", "US debt downgrade", 2),
    ("2015-08-24", "China 'Black Monday'", 2),
    ("2016-06-24", "Brexit vote", 2),
    ("2016-11-08", "US election shock", 1),
    ("2018-02-05", "Volmageddon", 2),
    ("2018-12-24", "Q4-2018 selloff", 2),
    ("2020-02-24", "COVID-19 crash", 3),
    ("2020-03-16", "COVID circuit breakers", 3),
    ("2022-02-24", "Russia invades Ukraine", 3),
    ("2022-06-13", "CPI / rate-shock selloff", 2),
    ("2023-03-10", "SVB / banking stress", 2),
]

def event_shock(index_like, events, half_life=15):
    days = np.arange(len(index_like))
    shock = np.zeros(len(index_like), dtype=float)
    decay = np.log(2) / half_life
    pos = {d: i for i, d in enumerate(index_like)}
    for date, _lbl, sev in events:
        ts = pd.Timestamp(date)
        loc = index_like.searchsorted(ts)
        if loc >= len(index_like):
            continue
        shock += sev * np.exp(-decay * np.clip(days - loc, 0, None)) * (days >= loc)
    return pd.Series(shock, index=index_like)

geo_shock = event_shock(feat.index, GEO_EVENTS)

# --- (3) Stock-specific event proxies from price & volume ------------------
close, vol = prices["Close"], prices["Volume"]
logret = np.log(close).diff()
vol_z = ((vol - vol.rolling(63).mean()) / (vol.rolling(63).std() + 1e-9))     # abnormal volume
overnight_gap = (prices["Open"] / close.shift() - 1.0)                        # gap vs prior close
move_z = (logret / (logret.rolling(63).std() + 1e-9))                        # abnormal daily move

def event_proximity(ticker, index_like):
    '''Days-to-nearest scheduled event (earnings/dividends) -> 'event soon' flag.'''
    soon = pd.Series(0.0, index=index_like)
    try:
        import yfinance as yf
        tk = yf.Ticker(ticker)
        dates = []
        try:
            ed = tk.get_earnings_dates(limit=40)
            if ed is not None and len(ed):
                dates += list(pd.to_datetime(ed.index).tz_localize(None))
        except Exception:
            pass
        try:
            div = tk.dividends
            if div is not None and len(div):
                dates += list(pd.to_datetime(div.index).tz_localize(None))
        except Exception:
            pass
        for d in dates:
            loc = index_like.searchsorted(d)
            for j in range(max(0, loc - 3), min(len(index_like), loc + 1)):
                soon.iloc[j] = 1.0          # flag the few days before a known event
    except Exception as e:
        print(f"[warn] event calendar unavailable ({e}); using 0.")
    return soon

event_soon = event_proximity(TICKER, feat.index)

# --- Optional plug-in for your own news-sentiment feed ---------------------
def news_sentiment_hook(index_like):
    '''Return a per-date sentiment score in [-1, 1]. Default: neutral (0).
    Replace the body with your own feed (e.g. a CSV of dated sentiment).'''
    return pd.Series(0.0, index=index_like)

news_sent = news_sentiment_hook(feat.index)

# --- Assemble the event/news/geopolitics features --------------------------
EVENTS = pd.DataFrame({
    "evt_vix_z":     ((vix - vix.rolling(252, min_periods=60).mean())
                      / (vix.rolling(252, min_periods=60).std() + 1e-9)).reindex(feat.index),
    "evt_vix_chg":   vix.pct_change(5).reindex(feat.index),
    "evt_geoshock":  geo_shock,
    "evt_vol_z":     vol_z.reindex(feat.index),
    "evt_gap":       overnight_gap.reindex(feat.index),
    "evt_move_z":    move_z.reindex(feat.index),
    "evt_soon":      event_soon.reindex(feat.index),
    "evt_news":      news_sent.reindex(feat.index),
}).replace([np.inf, -np.inf], 0.0).fillna(0.0)

EVENT_COLS = list(EVENTS.columns)
for col in EVENT_COLS:
    feat[col] = EVENTS[col]
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS + EVENT_COLS))
print(f"Added {len(EVENT_COLS)} event/news/geopolitics features. Total features: {len(FEATURE_COLS)}")
EVENTS.tail()


In [ ]:
# --- Labels: sign of the H-DAY-AHEAD return with a volatility-scaled dead-band
H = CONFIG["horizon"]
fwd_ret  = np.log(feat["close"].shift(-H) / feat["close"])   # H-day forward return
next_ret = feat["log_ret"].shift(-1)                         # 1-day return (for P&L)

# Dead-band scales with sqrt(H): a 5-day move is ~sqrt(5) larger than a 1-day one
band = CONFIG["deadband"] * feat["log_ret"].rolling(21).std() * np.sqrt(H)

label = pd.Series(0, index=feat.index)          # 0 = Flat
label[fwd_ret >  band] = 1                       # 1 = Long
label[fwd_ret < -band] = -1                      # -1 = Short

feat2 = feat.copy()
feat2["label"] = label
feat2["next_ret"] = next_ret

data = feat2.dropna().copy()
print(f"Usable samples: {len(data)}  |  horizon = {H} days")
print("Class balance (share):")
print((data["label"].value_counts(normalize=True)
       .rename({1: "Long", 0: "Flat", -1: "Short"}).round(3)))


<a id="datasets"></a>
## 8. Part II — Building the datasets: tabular **and** sequences

Different model families need different input shapes:

- **Tabular** `(samples, features)` — for the **MLP** and the **classical
  baselines** (Logistic Regression, Random Forest, Gradient Boosting). Each row
  is a snapshot of today's features.
- **Sequence** `(samples, lookback, features)` — for the **LSTM, GRU, 1D-CNN and
  Transformer**, which read a rolling *window* of the last `lookback` days.

Both views share the **same target day**, the **same chronological split**, and a
scaler fit **only on the training portion** (no look-ahead).


In [ ]:
def make_datasets(frame, feature_cols, cfg, train_frac=0.70, val_frac=0.85):
    '''Return aligned tabular + sequence datasets with a chronological split.'''
    L = cfg["lookback"]
    Xraw = frame[feature_cols].values.astype("float32")   # (n, F)
    y    = (frame["label"].values + 1).astype("int64")    # {-1,0,1}->{0,1,2}
    nr   = frame["next_ret"].values.astype("float32")
    vol  = frame["vol_21"].values.astype("float32")
    dates = frame.index
    n = len(frame)

    idx = np.arange(L - 1, n)                 # valid target rows (need L-1 history)
    n_valid = len(idx)
    i_tr = int(train_frac * n_valid)
    i_val = int(val_frac * n_valid)

    # Scaler fit ONLY on rows available before the validation set starts
    split_row = idx[i_tr]
    scaler = StandardScaler().fit(Xraw[:split_row])
    Xs = scaler.transform(Xraw).astype("float32")

    # Tabular (one row per target) and sequences (window ending at the target)
    X_tab = Xs[idx]
    X_seq = np.stack([Xs[t - L + 1: t + 1] for t in idx]).astype("float32")
    yv, nrv, volv, dv = y[idx], nr[idx], vol[idx], dates[idx]

    sl = {"tr": slice(0, i_tr), "val": slice(i_tr, i_val), "te": slice(i_val, None)}
    d = {"scaler": scaler, "n_features": Xs.shape[1], "lookback": L}
    for k, s in sl.items():
        d[f"Xtab_{k}"] = X_tab[s]
        d[f"Xseq_{k}"] = X_seq[s]
        d[f"y_{k}"]    = yv[s]
    d["test_dates"]    = dv[sl["te"]]
    d["test_next_ret"] = nrv[sl["te"]]
    d["test_vol"]      = volv[sl["te"]]
    return d

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

D = make_datasets(data, FEATURE_COLS, CONFIG)

# One-hot targets (for label smoothing) + per-sample weights (class imbalance)
n_classes = 3
classes = np.unique(D["y_tr"])
cw = compute_class_weight("balanced", classes=classes, y=D["y_tr"])
class_weight = {int(c): float(w) for c, w in zip(classes, cw)}
sw_tr = np.array([class_weight[int(c)] for c in D["y_tr"]], dtype="float32")

y_tr_oh  = keras.utils.to_categorical(D["y_tr"],  n_classes)
y_val_oh = keras.utils.to_categorical(D["y_val"], n_classes)

print(f"Features: {D['n_features']} | lookback: {D['lookback']}")
print(f"Tabular  train/val/test: {len(D['Xtab_tr'])}/{len(D['Xtab_val'])}/{len(D['Xtab_te'])}")
print(f"Sequence shape (train): {D['Xseq_tr'].shape}")
print("Class weights:", {(-1,0,1)[k]: round(v,2) for k,v in class_weight.items()})


<a id="zoo"></a>
## 9. Part II — A zoo of prediction models

We compare several **prediction models** on exactly the same data and the same
backtest, so we can judge which architecture actually helps.

### Neural networks (implemented in Keras)

| Model | Idea | Strength for markets |
|-------|------|----------------------|
| **MLP** | fully-connected net on today's features | simple, fast, strong baseline |
| **LSTM** | recurrent net with gated memory | captures temporal dependence in return sequences |
| **GRU** | lighter recurrent net (fewer gates) | similar to LSTM, faster, less overfitting |
| **1D-CNN** | temporal convolutions over the window | detects local patterns (momentum bursts, reversals) |
| **Transformer** | self-attention over the window | weighs *which past days* matter most |

### Classical baselines (scikit-learn)

| Model | Idea |
|-------|------|
| **Logistic Regression** | linear probabilistic classifier — the sanity check |
| **Random Forest** | bagged decision trees, robust to noise |
| **Gradient Boosting** | boosted trees, often the best tabular learner |

A recurring lesson in quantitative finance: on noisy tabular signals, **simple
models frequently match or beat deep nets**. Including baselines keeps us honest.
All models output calibrated 3-class probabilities `[P(Short), P(Flat), P(Long)]`
that feed the *same* position-sizing rule.


In [ ]:
from tensorflow.keras import regularizers

def _compile(model):
    model.compile(
        optimizer=keras.optimizers.Adam(5e-4),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"],
    )
    return model

def build_mlp(n_features, dropout=0.45, l2=3e-4):
    reg = regularizers.l2(l2)
    return _compile(keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(32, activation="relu", kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Dropout(dropout),
        layers.Dense(16, activation="relu", kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Dropout(dropout),
        layers.Dense(n_classes, activation="softmax"),
    ]))

def build_lstm(seq_len, n_features, units=32, dropout=0.3, l2=3e-4):
    reg = regularizers.l2(l2)
    return _compile(keras.Sequential([
        layers.Input(shape=(seq_len, n_features)),
        layers.LSTM(units, dropout=dropout, kernel_regularizer=reg),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu", kernel_regularizer=reg),
        layers.Dense(n_classes, activation="softmax"),
    ]))

def build_gru(seq_len, n_features, units=32, dropout=0.3, l2=3e-4):
    reg = regularizers.l2(l2)
    return _compile(keras.Sequential([
        layers.Input(shape=(seq_len, n_features)),
        layers.GRU(units, dropout=dropout, kernel_regularizer=reg),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu", kernel_regularizer=reg),
        layers.Dense(n_classes, activation="softmax"),
    ]))

def build_cnn(seq_len, n_features, filters=32, dropout=0.3, l2=3e-4):
    reg = regularizers.l2(l2)
    return _compile(keras.Sequential([
        layers.Input(shape=(seq_len, n_features)),
        layers.Conv1D(filters, 3, activation="relu", padding="causal", kernel_regularizer=reg),
        layers.Conv1D(filters // 2, 3, activation="relu", padding="causal", kernel_regularizer=reg),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu", kernel_regularizer=reg),
        layers.Dense(n_classes, activation="softmax"),
    ]))

def build_transformer(seq_len, n_features, d_model=32, heads=2, dropout=0.3, l2=3e-4):
    reg = regularizers.l2(l2)
    inp = layers.Input(shape=(seq_len, n_features))
    x = layers.Dense(d_model)(inp)                       # project to d_model
    # --- self-attention block (pre-norm residual) ---
    a = layers.LayerNormalization()(x)
    a = layers.MultiHeadAttention(num_heads=heads, key_dim=d_model // heads,
                                  dropout=dropout)(a, a)
    x = layers.Add()([x, a])
    # --- feed-forward block ---
    f = layers.LayerNormalization()(x)
    f = layers.Dense(d_model, activation="relu", kernel_regularizer=reg)(f)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(d_model)(f)
    x = layers.Add()([x, f])
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(16, activation="relu", kernel_regularizer=reg)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return _compile(keras.Model(inp, out))

print("Model builders ready: MLP, LSTM, GRU, CNN, Transformer.")


In [ ]:
# --- Train every neural network on its proper input shape -------------------
def unweighted_acc(model, X, y_int):
    return float((model.predict(X, verbose=0).argmax(1) == y_int).mean())

def train_keras(model, Xtr, Xval, epochs=120, batch=128):
    cbs = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=12,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=6, min_lr=1e-5),
    ]
    return model.fit(Xtr, y_tr_oh, validation_data=(Xval, y_val_oh),
                     sample_weight=sw_tr, epochs=epochs, batch_size=batch,
                     callbacks=cbs, verbose=0)

nn_specs = {
    "MLP":         (lambda: build_mlp(D["n_features"]),                       "tab"),
    "LSTM":        (lambda: build_lstm(D["lookback"], D["n_features"]),       "seq"),
    "GRU":         (lambda: build_gru(D["lookback"], D["n_features"]),        "seq"),
    "CNN":         (lambda: build_cnn(D["lookback"], D["n_features"]),        "seq"),
    "Transformer": (lambda: build_transformer(D["lookback"], D["n_features"]),"seq"),
}

results = {}          # name -> dict(proba_te, val_acc, gap, kind)
histories = {}
for name, (factory, kind) in nn_specs.items():
    Xtr = D[f"X{kind}_tr"]; Xval = D[f"X{kind}_val"]; Xte = D[f"X{kind}_te"]
    keras.utils.set_random_seed(SEED)
    model = factory()
    h = train_keras(model, Xtr, Xval)
    tr_acc = unweighted_acc(model, Xtr, D["y_tr"])
    va_acc = unweighted_acc(model, Xval, D["y_val"])
    results[name] = {
        "proba_te": model.predict(Xte, verbose=0),
        "val_acc": va_acc, "gap": tr_acc - va_acc, "kind": "NN",
        "model": model, "dkind": kind,
    }
    histories[name] = h
    print(f"{name:12s} | epochs {len(h.history['loss']):3d} | "
          f"train acc {tr_acc:.3f} | val acc {va_acc:.3f} | gap {tr_acc-va_acc:+.3f}")


In [ ]:
# --- Classical baselines on the tabular features ---------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

def proba_3(clf, X):
    '''Return probabilities as [P(0),P(1),P(2)] regardless of clf.classes_ order.'''
    p = clf.predict_proba(X)
    out = np.zeros((len(X), 3), dtype="float32")
    for j, c in enumerate(clf.classes_):
        out[:, int(c)] = p[:, j]
    return out

classical = {
    "LogReg":       LogisticRegression(max_iter=2000, class_weight="balanced"),
    # Heavily regularized so the train-val gap stays small (no memorising):
    # shallow trees, large leaves, few features per split.
    "RandomForest": RandomForestClassifier(n_estimators=400, max_depth=4,
                                           min_samples_leaf=80, max_features="sqrt",
                                           class_weight="balanced",
                                           random_state=SEED, n_jobs=-1),
    # Early stopping on an internal validation split stops boosting before it
    # overfits; small trees + strong L2 + slow learning rate keep the gap down.
    "GradBoost":    HistGradientBoostingClassifier(max_depth=2, max_leaf_nodes=15,
                                                   learning_rate=0.03, max_iter=600,
                                                   min_samples_leaf=80, l2_regularization=5.0,
                                                   early_stopping=True, validation_fraction=0.15,
                                                   n_iter_no_change=15, random_state=SEED),
}

for name, clf in classical.items():
    if name == "GradBoost":
        clf.fit(D["Xtab_tr"], D["y_tr"], sample_weight=sw_tr)
    else:
        clf.fit(D["Xtab_tr"], D["y_tr"])
    tr_acc = float((clf.predict(D["Xtab_tr"]) == D["y_tr"]).mean())
    va_acc = float((clf.predict(D["Xtab_val"]) == D["y_val"]).mean())
    results[name] = {
        "proba_te": proba_3(clf, D["Xtab_te"]),
        "val_acc": va_acc, "gap": tr_acc - va_acc, "kind": "Classical",
        "model": clf, "dkind": "tab",
    }
    print(f"{name:12s} | train acc {tr_acc:.3f} | val acc {va_acc:.3f} | gap {tr_acc-va_acc:+.3f}")


In [ ]:
# --- Overfitting check across all models: validation accuracy & train-val gap
names = list(results.keys())
val_accs = [results[n]["val_acc"] for n in names]
gaps     = [results[n]["gap"] for n in names]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
colors = ["#4C72B0" if results[n]["kind"] == "NN" else "#DD8452" for n in names]
ax[0].bar(names, val_accs, color=colors)
ax[0].axhline(1/3, color="gray", ls="--", label="random (1/3)")
ax[0].set_title("Validation accuracy by model"); ax[0].set_ylabel("Accuracy")
ax[0].tick_params(axis="x", rotation=45); ax[0].legend()

ax[1].bar(names, gaps, color=colors)
ax[1].axhline(0.05, color="crimson", ls="--", label="overfit warning (~0.05)")
ax[1].set_title("Train − Validation accuracy gap (overfitting)")
ax[1].set_ylabel("Gap"); ax[1].tick_params(axis="x", rotation=45); ax[1].legend()
plt.tight_layout(); plt.show()


<a id="backtest"></a>
## 10. Part III — Backtesting all models with the new levers

### From probabilities to a position

The softmax probabilities become a **continuous target exposure**:

$$
\text{conviction}_t = \big(p^{\text{long}}_t - p^{\text{short}}_t\big)\cdot g
\; + \; b,
$$

where $g$ = `gain` amplifies the signal and $b$ = `long_bias` tilts the book
toward being invested. If `allow_short` is `False` the conviction is floored at
$0$ (**long-or-flat**). We then multiply by a **volatility-target scale**
$\min(\sigma_{\text{target}}/\sigma_t,\ \text{max\_leverage})$ — the BS $\sigma$
controlling risk — apply the exposure to **tomorrow's** return, and subtract
**transaction costs** proportional to turnover.

Every model is run through the **identical** pipeline, then ranked on a single
leaderboard by **Sharpe, return and drawdown** versus Buy & Hold.


In [ ]:
def probs_to_position(proba, realized_vol, cfg):
    p_short, p_long = proba[:, 0], proba[:, 2]
    conviction = (p_long - p_short) * cfg["gain"] + cfg["long_bias"]
    low = -1.0 if cfg["allow_short"] else 0.0
    conviction = np.clip(conviction, low, 1.0)

    rv = np.where(realized_vol > 1e-6, realized_vol, np.nan)
    scale = np.clip(cfg["vol_target"] / rv, 0.0, cfg["max_leverage"])
    scale = pd.Series(scale).ffill().fillna(1.0).values

    pos = conviction * scale
    lo = -cfg["max_leverage"] if cfg["allow_short"] else 0.0
    return np.clip(pos, lo, cfg["max_leverage"])

def run_backtest(dates, next_rets, position, cost_bps):
    turnover = np.abs(np.diff(position, prepend=0.0))
    costs = turnover * (cost_bps / 1e4)
    strat = position * next_rets - costs
    df = pd.DataFrame({"position": position, "strategy": strat, "buy_hold": next_rets},
                      index=dates)
    df["equity_strategy"] = np.exp(df["strategy"].cumsum())
    df["equity_buyhold"]  = np.exp(df["buy_hold"].cumsum())
    return df

def performance_stats(returns, periods=252):
    r = pd.Series(returns).dropna()
    if len(r) == 0:
        return {}
    equity = np.exp(r.cumsum())
    dd = equity / equity.cummax() - 1
    downside = r[r < 0].std() * np.sqrt(periods)
    return {
        "total":   np.exp(r.sum()) - 1,
        "ann_ret": np.exp(r.mean() * periods) - 1,
        "ann_vol": r.std() * np.sqrt(periods),
        "sharpe":  (r.mean() / (r.std() + 1e-12)) * np.sqrt(periods),
        "sortino": (r.mean() * periods) / (downside + 1e-12),
        "max_dd":  dd.min(),
        "calmar":  (np.exp(r.mean() * periods) - 1) / (abs(dd.min()) + 1e-12),
        "win":     (r > 0).mean(),
    }

print("Backtest engine ready.")


In [ ]:
# --- Run the backtest for every model and build the leaderboard ------------
rows = []
for name, info in results.items():
    pos = probs_to_position(info["proba_te"], D["test_vol"], CONFIG)
    bt_m = run_backtest(D["test_dates"], D["test_next_ret"], pos, CONFIG["cost_bps"])
    info["bt"] = bt_m
    s = performance_stats(bt_m["strategy"])
    rows.append({
        "Model": name, "Type": info["kind"],
        "Val acc": info["val_acc"], "Gap": info["gap"],
        "Total ret": s["total"], "Ann ret": s["ann_ret"],
        "Sharpe": s["sharpe"], "Max DD": s["max_dd"],
        "Calmar": s["calmar"], "Win rate": s["win"],
    })

# Buy & Hold benchmark (same test window)
any_bt = next(iter(results.values()))["bt"]
sbh = performance_stats(any_bt["buy_hold"])
rows.append({"Model": "Buy & Hold", "Type": "Benchmark", "Val acc": np.nan, "Gap": np.nan,
             "Total ret": sbh["total"], "Ann ret": sbh["ann_ret"], "Sharpe": sbh["sharpe"],
             "Max DD": sbh["max_dd"], "Calmar": sbh["calmar"], "Win rate": sbh["win"]})

board = pd.DataFrame(rows).sort_values("Sharpe", ascending=False).reset_index(drop=True)

fmt = board.copy()
for c in ["Val acc", "Gap", "Total ret", "Ann ret", "Max DD", "Win rate"]:
    fmt[c] = fmt[c].map(lambda v: "" if pd.isna(v) else f"{v:.1%}")
for c in ["Sharpe", "Calmar"]:
    fmt[c] = fmt[c].map(lambda v: f"{v:.2f}")
print("OUT-OF-SAMPLE LEADERBOARD (ranked by Sharpe):\n")
print(fmt.to_string(index=False))


<a id="results"></a>
## 11. Part III — Results & charts

First, all strategies on one equity chart against Buy & Hold; then a detailed
breakdown of the **best *trustworthy* model** — the highest Sharpe **among the
models that did not overfit** (train-val gap ≤ 0.08). A dazzling return from a
model that memorised the training set is not tradeable, so we exclude it.


In [ ]:
# --- Equity curves: every model vs Buy & Hold ------------------------------
plt.figure(figsize=(13, 6))
for name, info in results.items():
    plt.plot(info["bt"].index, info["bt"]["equity_strategy"], lw=1.3, label=name)
plt.plot(any_bt.index, any_bt["equity_buyhold"], lw=2.5, color="black",
         ls="--", label="Buy & Hold")
plt.title("Out-of-sample equity curves — all models (growth of 1)")
plt.ylabel("Equity"); plt.xlabel("Date")
plt.legend(ncol=3, fontsize=9); plt.tight_layout(); plt.show()


In [ ]:
# --- Pick the best TRUSTWORTHY model ---------------------------------------
# A high Sharpe means nothing if it comes from an overfit model, so we only
# consider models whose train-val gap is small, then take the best Sharpe
# among those. This avoids crowning an overfitter (e.g. a boosted-tree that
# memorised the training set) as the winner.
GAP_MAX = 0.08
cand = board[board["Type"] != "Benchmark"].copy()
trust = cand[cand["Gap"] <= GAP_MAX]
if len(trust):
    best_name = trust.sort_values("Sharpe", ascending=False).iloc[0]["Model"]
    print(f"Best TRUSTWORTHY model (gap <= {GAP_MAX:.2f}, ranked by Sharpe): {best_name}")
else:
    best_name = cand.sort_values("Gap").iloc[0]["Model"]
    print(f"No model under gap {GAP_MAX:.2f}; falling back to the lowest-gap model: {best_name}")

overfit = cand[cand["Gap"] > GAP_MAX]["Model"].tolist()
if overfit:
    print(f"Excluded as overfitting (gap > {GAP_MAX:.2f}): {', '.join(overfit)}")

bt = results[best_name]["bt"]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.plot(bt.index, bt["equity_strategy"], lw=1.8, label=f"{best_name} strategy")
ax.plot(bt.index, bt["equity_buyhold"], lw=1.4, alpha=0.8, label="Buy & Hold")
ax.set_title(f"Equity curve — {best_name}"); ax.set_ylabel("Equity"); ax.legend()

ax = axes[0, 1]
eq = bt["equity_strategy"]; dd = eq / eq.cummax() - 1
ax.fill_between(bt.index, dd, 0, color="crimson", alpha=0.4)
ax.set_title("Strategy drawdown"); ax.set_ylabel("Drawdown")

ax = axes[1, 0]
ax.plot(bt.index, bt["position"], lw=1.0, color="teal")
ax.axhline(0, color="gray", lw=0.8)
lo = -CONFIG["max_leverage"] if CONFIG["allow_short"] else 0.0
ax.set_ylim(lo - 0.1, CONFIG["max_leverage"] + 0.1)
ax.set_title("Exposure over time"); ax.set_ylabel("Position")

ax = axes[1, 1]
ax.hist(bt["strategy"], bins=40, alpha=0.7, color="slateblue")
ax.axvline(bt["strategy"].mean(), color="black", ls="--",
           label=f"mean = {bt['strategy'].mean():.4f}")
ax.set_title("Distribution of daily strategy returns")
ax.set_xlabel("Daily log-return"); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# --- Rolling 63-day annualized Sharpe: best model vs Buy & Hold -------------
def rolling_sharpe(x, w=63):
    r = x.rolling(w)
    return (r.mean() / (r.std() + 1e-12)) * np.sqrt(252)

plt.figure(figsize=(12, 4))
plt.plot(bt.index, rolling_sharpe(bt["strategy"]), lw=1.4, label=best_name)
plt.plot(bt.index, rolling_sharpe(bt["buy_hold"]), lw=1.2, alpha=0.7, label="Buy & Hold")
plt.axhline(0, color="gray", lw=0.8)
plt.axhline(1, color="green", ls="--", lw=0.8, label="Sharpe = 1")
plt.title(f"Rolling 63-day annualized Sharpe — {best_name} vs Buy & Hold")
plt.ylabel("Sharpe"); plt.legend(); plt.tight_layout(); plt.show()


<a id="importance"></a>
## 11b. Part III — Which signals matter? Permutation importance

Now that the model bank is rich (BS/volatility features, 12 trading strategies,
chart patterns, the analogue memory, and the events/news/geopolitics block), the
natural question is: **which of these does the best model actually use?**

We answer it with **permutation importance**, a model-agnostic method: shuffle one
feature across the test set and measure how much the model's **accuracy drops**.
A big drop ⇒ the model relied on that feature; ~zero ⇒ it ignored it (or the
feature is noise). We show the top individual features and the **importance summed
by feature family**, so you can see whether the strategies / patterns / memory /
events are pulling their weight.


In [ ]:
# --- Permutation importance for the best trustworthy model -----------------
def feature_group(name):
    if name.startswith("strat_"): return "Trading strategies"
    if name.startswith("pat_"):   return "Chart patterns"
    if name.startswith("mem_"):   return "Analogue memory"
    if name.startswith("evt_"):   return "Events/news/geo"
    return "BS / volatility"

def permutation_importance(name, n_repeats=5):
    info = results[name]
    Xte = (D["Xseq_te"] if info["dkind"] == "seq" else D["Xtab_te"]).copy()
    yte = D["y_te"]
    if info["kind"] == "NN":
        predict = lambda X: info["model"].predict(X, verbose=0).argmax(1)
    else:
        predict = lambda X: info["model"].predict(X)
    base = (predict(Xte) == yte).mean()
    rng = np.random.default_rng(SEED)
    imp = np.zeros(len(FEATURE_COLS))
    for j in range(len(FEATURE_COLS)):
        drops = []
        for _ in range(n_repeats):
            Xp = Xte.copy()
            perm = rng.permutation(len(Xte))
            if info["dkind"] == "seq":
                Xp[:, :, j] = Xte[perm][:, :, j]
            else:
                Xp[:, j] = Xte[perm][:, j]
            drops.append(base - (predict(Xp) == yte).mean())
        imp[j] = np.mean(drops)
    return pd.Series(imp, index=FEATURE_COLS).sort_values(ascending=False)

imp = permutation_importance(best_name)
print(f"Permutation importance for the best trustworthy model: {best_name}")

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# (a) Top-20 individual features, coloured by family
top = imp.head(20)[::-1]
palette = {"Trading strategies": "#4C72B0", "Chart patterns": "#DD8452",
           "Analogue memory": "#55A868", "Events/news/geo": "#C44E52",
           "BS / volatility": "#8172B3"}
bar_colors = [palette[feature_group(f)] for f in top.index]
ax[0].barh(top.index, top.values, color=bar_colors)
ax[0].axvline(0, color="gray", lw=0.8)
ax[0].set_title(f"Top-20 features — {best_name}")
ax[0].set_xlabel("Accuracy drop when shuffled")

# (b) Importance summed by family
grp = imp.groupby(feature_group).sum().sort_values()
ax[1].barh(grp.index, grp.values, color=[palette[g] for g in grp.index])
ax[1].axvline(0, color="gray", lw=0.8)
ax[1].set_title("Importance summed by feature family")
ax[1].set_xlabel("Total accuracy drop")
plt.tight_layout(); plt.show()

print("\\nImportance by family (share of total positive importance):")
pos = grp.clip(lower=0)
print((pos / (pos.sum() + 1e-12)).sort_values(ascending=False).map(lambda v: f"{v:.1%}").to_string())


<a id="budget"></a>
## 11c. Part III — Practical report: a **€200** budget

Enough abstractions — what would **€200** have done? We take the best model's
**out-of-sample daily returns** (already net of costs), grow €200 along that path,
and report it in euros: final value, profit, worst drawdown, and a month-by-month
breakdown.

Then, because the single historical path is just *one* possible outcome, we run a
**Monte-Carlo bootstrap**: we resample the strategy's daily returns thousands of
times (in short blocks, to keep short-term dynamics) to see the **distribution of
what €200 could become** — the realistic *range* of profits, not a single number.

> ⚠️ This assumes you can actually trade the model's (possibly leveraged,
> fractional) exposure with €200 and that the return distribution repeats. Real
> derivatives add margin calls, expiry and path risk. Treat the numbers as
> *illustrative*, not a promise.


In [ ]:
BUDGET = 200.0   # euros

def budget_report(daily_log_ret, budget=BUDGET, label="strategy", periods=252):
    r = pd.Series(daily_log_ret).dropna()
    eq = budget * np.exp(r.cumsum())
    final = float(eq.iloc[-1]); profit = final - budget
    dd_eur = float((eq - eq.cummax()).min())
    ddp = float((eq / eq.cummax() - 1).min())
    years = len(r) / periods
    cagr = (final / budget) ** (1 / max(years, 1e-9)) - 1
    best_day = float((np.exp(r.max()) - 1)); worst_day = float((np.exp(r.min()) - 1))
    print(f"=== €{budget:.0f} budget report — {label} ===")
    print(f"Period            : {r.index[0].date()} -> {r.index[-1].date()}  ({years:.1f} yrs)")
    print(f"Final value       : €{final:,.2f}")
    print(f"Profit / loss     : €{profit:,.2f}  ({profit/budget:+.1%})")
    print(f"CAGR              : {cagr:+.1%}")
    print(f"Worst drawdown    : €{dd_eur:,.2f}  ({ddp:.1%})")
    print(f"Best / worst day  : {best_day:+.2%} / {worst_day:+.2%}")
    return eq

eq_eur = budget_report(results[best_name]["bt"]["strategy"], label=best_name)

# Monthly profit/loss table (in euros)
monthly = (np.exp(pd.Series(results[best_name]["bt"]["strategy"]).resample("ME").sum()) - 1)
monthly_eur = eq_eur.resample("ME").last().diff().fillna(eq_eur.iloc[0] - BUDGET)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(eq_eur.index, eq_eur.values, lw=1.8, color="#4C72B0")
ax[0].axhline(BUDGET, color="gray", ls="--", label=f"start €{BUDGET:.0f}")
ax[0].set_title(f"Growth of €{BUDGET:.0f} — {best_name}"); ax[0].set_ylabel("Euro"); ax[0].legend()
colors = ["#55A868" if v >= 0 else "#C44E52" for v in monthly_eur.values]
ax[1].bar(range(len(monthly_eur)), monthly_eur.values, color=colors)
ax[1].axhline(0, color="gray", lw=0.8)
ax[1].set_title("Monthly profit / loss (€)"); ax[1].set_xlabel("Month #"); ax[1].set_ylabel("Euro")
plt.tight_layout(); plt.show()


In [ ]:
# --- Monte-Carlo: distribution of what €200 could become -------------------
def monte_carlo_budget(daily_log_ret, budget=BUDGET, n_sims=5000, block=5, seed=SEED):
    r = pd.Series(daily_log_ret).dropna().values
    n = len(r)
    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(n / block))
    finals = np.empty(n_sims)
    for i in range(n_sims):
        starts = rng.integers(0, max(n - block, 1), size=n_blocks)
        path = np.concatenate([r[s:s + block] for s in starts])[:n]
        finals[i] = budget * np.exp(path.sum())
    return finals

finals = monte_carlo_budget(results[best_name]["bt"]["strategy"])
pct = np.percentile(finals, [5, 25, 50, 75, 95])
p_profit = float((finals > BUDGET).mean())
p_loss50 = float((finals < BUDGET * 0.5).mean())

print(f"Monte-Carlo of €{BUDGET:.0f} over the test horizon ({len(finals):,} simulated paths):")
print(f"  5th  percentile : €{pct[0]:,.2f}")
print(f"  25th percentile : €{pct[1]:,.2f}")
print(f"  Median          : €{pct[2]:,.2f}")
print(f"  75th percentile : €{pct[3]:,.2f}")
print(f"  95th percentile : €{pct[4]:,.2f}")
print(f"  P(ending in profit)     : {p_profit:.1%}")
print(f"  P(losing >50% of budget): {p_loss50:.1%}")

plt.figure(figsize=(12, 4.5))
plt.hist(finals, bins=60, color="#8172B3", alpha=0.75)
plt.axvline(BUDGET, color="black", ls="--", label=f"start €{BUDGET:.0f}")
plt.axvline(pct[2], color="green", ls="-", label=f"median €{pct[2]:.0f}")
plt.axvline(pct[0], color="red", ls=":", label=f"5th pct €{pct[0]:.0f}")
plt.axvline(pct[4], color="red", ls=":", label=f"95th pct €{pct[4]:.0f}")
plt.title(f"What could €{BUDGET:.0f} become? — Monte-Carlo of {best_name} daily returns")
plt.xlabel("Final value (€)"); plt.ylabel("Frequency"); plt.legend(); plt.tight_layout(); plt.show()


<a id="portfolio"></a>
## Part IV — Extending the NN to a **portfolio of several derivatives**

So far we traded one underlying. We now let the network manage a **portfolio of
several derivatives**: a **leveraged directional position on each of several
underlyings**. A position of, say, `+1.4` on an asset is exactly the exposure a
**futures / CFD or a delta-one derivative** gives you — the same `max_leverage`
gearing already in `CONFIG` — and it could equally be expressed as a
Black–Scholes-priced call/put using the Greeks from Part I.

**Design.**
1. **Universe** of underlyings (diversified: equities, bonds, gold, tech).
2. **One shared network** trained on the **pooled** BS/volatility features of all
   assets — it learns a *general* directional signal that transfers across
   instruments (more data, less overfitting than one model per asset).
3. Each asset's probabilities → a **derivative position** (leveraged) via the same
   `probs_to_position` rule.
4. Positions are combined with **inverse-volatility (risk-parity) weights**, so
   calmer instruments get more capital and the book targets a stable risk level —
   the diversification that a single-asset strategy cannot provide.


In [ ]:
# --- 1) Build the derivative universe --------------------------------------
UNIVERSE = ["SPY", "QQQ", "TLT", "GLD"]     # equities / tech / long bonds / gold

def load_universe(tickers, start, end):
    uni = {}
    for t in tickers:
        p = load_prices(t, start, end)
        if "SIMULATED" in p.attrs.get("source", ""):
            uni = {}                         # offline -> use a correlated synthetic panel
            break
        uni[t] = p
    if not uni:
        print("[warn] building a correlated SYNTHETIC panel for the universe.")
        n = 252 * 12
        rng = np.random.default_rng(SEED)
        drifts = {"SPY": 0.08, "QQQ": 0.11, "TLT": 0.02, "GLD": 0.05}
        vols   = {"SPY": 0.16, "QQQ": 0.22, "TLT": 0.12, "GLD": 0.15}
        base = rng.standard_normal(n)        # a shared market factor
        idx = pd.bdate_range(start=start, periods=n + 1)
        for t in tickers:
            beta = 0.6 if t != "TLT" else -0.3
            shocks = beta * base + np.sqrt(max(1 - beta**2, 0.05)) * rng.standard_normal(n)
            lr = (drifts[t] - 0.5 * vols[t]**2) / 252 + vols[t] / np.sqrt(252) * shocks
            close = 100 * np.exp(np.concatenate([[0], np.cumsum(lr)]))
            rel = np.abs(rng.normal(0, 0.008, close.shape))
            uni[t] = pd.DataFrame({"Open": close, "High": close * (1 + rel),
                                   "Low": close * (1 - rel), "Close": close,
                                   "Volume": rng.lognormal(15, 0.4, close.shape)}, index=idx)
    return uni

uni = load_universe(UNIVERSE, START, END)
print("Universe:", list(uni.keys()))
for t, p in uni.items():
    print(f"  {t}: {len(p)} rows  {p.index[0].date()} -> {p.index[-1].date()}")


In [ ]:
# --- 2) Per-asset frames (compact BS/volatility features) + pooled split ---
def build_asset_frame(pr):
    f = build_features(pr)
    H = CONFIG["horizon"]
    fwd = np.log(f["close"].shift(-H) / f["close"])
    f["next_ret"] = f["log_ret"].shift(-1)
    band = CONFIG["deadband"] * f["log_ret"].rolling(21).std() * np.sqrt(H)
    lab = pd.Series(0, index=f.index)
    lab[fwd > band] = 1; lab[fwd < -band] = -1
    f["label"] = lab
    return f.dropna()

frames = {t: build_asset_frame(p) for t, p in uni.items()}
FEAT_P = [c for c in frames[UNIVERSE[0]].columns
          if c not in ("close", "log_ret", "label", "next_ret")]

# Chronological split by DATE (shared across all assets -> no look-ahead)
all_dates = np.array(sorted(set().union(*[f.index for f in frames.values()])))
d_tr  = all_dates[int(0.70 * len(all_dates))]
d_val = all_dates[int(0.85 * len(all_dates))]

def pool(split):
    Xs, ys = [], []
    for f in frames.values():
        if split == "tr":  m = f.index < d_tr
        elif split == "val": m = (f.index >= d_tr) & (f.index < d_val)
        else: m = f.index >= d_val
        Xs.append(f.loc[m, FEAT_P].values); ys.append((f.loc[m, "label"].values + 1))
    return np.vstack(Xs).astype("float32"), np.concatenate(ys).astype("int64")

Xtr_p, ytr_p = pool("tr"); Xval_p, yval_p = pool("val")
scaler_p = StandardScaler().fit(Xtr_p)
Xtr_ps, Xval_ps = scaler_p.transform(Xtr_p), scaler_p.transform(Xval_p)

cw_p = compute_class_weight("balanced", classes=np.unique(ytr_p), y=ytr_p)
cwd_p = {int(c): float(w) for c, w in zip(np.unique(ytr_p), cw_p)}
sw_p = np.array([cwd_p[int(c)] for c in ytr_p], dtype="float32")
print(f"Pooled train {len(Xtr_p)} | val {len(Xval_p)} | features {len(FEAT_P)}")


In [ ]:
# --- 3) Train ONE shared network on the pooled multi-asset data ------------
keras.utils.set_random_seed(SEED)
port_model = build_mlp(len(FEAT_P))
port_model.fit(Xtr_ps, keras.utils.to_categorical(ytr_p, 3),
               validation_data=(Xval_ps, keras.utils.to_categorical(yval_p, 3)),
               sample_weight=sw_p, epochs=120, batch_size=256,
               callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=12,
                          restore_best_weights=True)], verbose=0)
va = float((port_model.predict(Xval_ps, verbose=0).argmax(1) == yval_p).mean())
print(f"Shared portfolio model trained. Pooled validation accuracy: {va:.3f}")


In [ ]:
# --- 4) Per-asset derivative positions + inverse-vol portfolio -------------
asset_ret, asset_pos = {}, {}
for t, f in frames.items():
    te = f.index >= d_val
    Xte = scaler_p.transform(f.loc[te, FEAT_P].values).astype("float32")
    proba = port_model.predict(Xte, verbose=0)
    pos = probs_to_position(proba, f.loc[te, "vol_21"].values, CONFIG)   # leveraged derivative
    bt_t = run_backtest(f.index[te], f.loc[te, "next_ret"].values, pos, CONFIG["cost_bps"])
    asset_ret[t] = bt_t["strategy"]; asset_pos[t] = pd.Series(pos, index=f.index[te])

R = pd.DataFrame(asset_ret).dropna()            # aligned per-asset strategy returns
VOL = pd.DataFrame({t: frames[t]["vol_21"] for t in frames}).reindex(R.index).ffill()
W = (1.0 / (VOL + 1e-9)); W = W.div(W.sum(axis=1), axis=0)   # inverse-vol weights
port_strategy = (W * R).sum(axis=1)                          # portfolio daily return

# Equal-weight buy & hold of the same universe (benchmark)
BH = pd.DataFrame({t: frames[t]["next_ret"] for t in frames}).reindex(R.index)
port_bh = BH.mean(axis=1)

port_stats = performance_stats(port_strategy)
bh_stats = performance_stats(port_bh)
comp = pd.DataFrame({"NN derivative portfolio": port_stats, "Equal-weight Buy&Hold": bh_stats})
for c in comp.columns:
    comp[c] = comp.index.map(lambda k: (f"{comp.loc[k, c]:.1%}" if k in
              ("total","ann_ret","ann_vol","max_dd","win") else f"{comp.loc[k, c]:.2f}"))
print("Multi-derivative portfolio — out-of-sample performance:\n")
print(comp.to_string())


In [ ]:
# --- Portfolio charts ------------------------------------------------------
eq_port = np.exp(port_strategy.cumsum()); eq_bh = np.exp(port_bh.cumsum())
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.plot(eq_port.index, eq_port, lw=1.9, label="NN derivative portfolio")
ax.plot(eq_bh.index, eq_bh, lw=1.4, alpha=0.8, label="Equal-weight Buy & Hold")
ax.set_title("Portfolio equity (growth of 1)"); ax.set_ylabel("Equity"); ax.legend()

ax = axes[0, 1]
for t in R.columns:
    ax.plot(R.index, np.exp(R[t].cumsum()), lw=1.2, label=t)
ax.set_title("Per-derivative equity (each sleeve)"); ax.legend(ncol=2, fontsize=8)

ax = axes[1, 0]
dd = eq_port / eq_port.cummax() - 1
ax.fill_between(eq_port.index, dd, 0, color="crimson", alpha=0.4)
ax.set_title("Portfolio drawdown"); ax.set_ylabel("Drawdown")

ax = axes[1, 1]
ax.stackplot(W.index, [W[t] for t in W.columns], labels=list(W.columns), alpha=0.8)
ax.set_title("Inverse-volatility weights over time"); ax.set_ylabel("Weight")
ax.legend(ncol=2, fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()


In [ ]:
# --- €200 report on the diversified derivative portfolio -------------------
eq_eur_port = budget_report(port_strategy, label="NN derivative portfolio")
finals_port = monte_carlo_budget(port_strategy)
pct_p = np.percentile(finals_port, [5, 50, 95])
print(f"\\nMonte-Carlo €{BUDGET:.0f} on the portfolio: "
      f"5th €{pct_p[0]:,.0f} | median €{pct_p[1]:,.0f} | 95th €{pct_p[2]:,.0f} | "
      f"P(profit) {float((finals_port > BUDGET).mean()):.1%}")

# Single asset vs diversified portfolio: the diversification effect
plt.figure(figsize=(12, 4.5))
plt.plot(eq_eur.index, eq_eur.values, lw=1.6, label=f"Single: {best_name} (€200)")
plt.plot(eq_eur_port.index, eq_eur_port.values, lw=1.9, label="Diversified derivative portfolio (€200)")
plt.axhline(BUDGET, color="gray", ls="--")
plt.title("€200 — single-asset strategy vs diversified derivative portfolio")
plt.ylabel("Euro"); plt.legend(); plt.tight_layout(); plt.show()


<a id="conclusions"></a>
## 13. Conclusions & caveats

**What we built.**
- A **Black–Scholes simulation** engine (GBM paths, closed-form pricer, Greeks).
- A **rich feature bank** feeding a whole **zoo of prediction models** (MLP, LSTM,
  GRU, 1D-CNN, Transformer + Logistic Regression, Random Forest, Gradient Boosting):
  BS/volatility features, the **signals of 12 classic trading strategies**
  (trend, mean-reversion, volatility/GARCH), **candlestick / chart pattern**
  detectors, an **analogue k-NN memory** of similar past days, and an
  **events / news / geopolitics** block (VIX, a curated event overlay, and
  stock-specific volume/gap/earnings proxies).
- A tunable **decision layer** (`CONFIG`) implementing the levers that fixed the
  earlier flat-equity result: a **multi-day horizon**, a **long-bias / long-only**
  regime, higher **gain** and **leverage**, and volatility-target sizing.
- A single **leaderboard + charts** ranking every model out-of-sample on Sharpe,
  return and drawdown against Buy & Hold.

**How to read the comparison.**
- The **validation-gap chart** shows which models overfit (big train−val gap).
- The **leaderboard** and **equity chart** show which models actually add value
  *after costs*. Watch for the classic result: **simple models (LogReg / boosting)
  often rival the deep nets** on noisy financial data.
- With a long-bias regime, beating Buy & Hold in a strong bull market is hard;
  the fair question is **risk-adjusted** performance (Sharpe, Calmar, drawdown).

**Honest caveats.**
- A single train/val/test split is optimistic — use **walk-forward / purged CV**.
- Edges are thin and costs bite; treat positive Sharpe as *illustrative*.
- No slippage, borrow, market impact or regime handling beyond a flat bps cost.

**Natural next steps.**
- **Ensemble** the models (average probabilities) — often more robust than any one.
- Predict **volatility** (a true BS input) and trade **options** with the Greeks.
- Add **implied volatility** / the vol surface as features.
- Replace the fixed probability-to-position map with **reinforcement learning**.

*Educational use only — not investment advice.*
